# Classifying Climate-Induced Disaster Impacts on Nepali Households Using Machine Learning
**ST7072CEM Machine Learning – Avin Amatya (Student ID 17624321)**

Dataset: *National Climate Change Survey 2022 (NCCS-II)*, National Statistics Office (NSO), Nepal – 6,508 households, 326 primary sampling units (PSUs), all seven provinces.

**Task:** predict whether a household experienced a **high or very high impact** from at least one climate-induced disaster in the past 25 years, using only household characteristics recorded *before* / independently of the impact.

**Models compared:** Logistic Regression, Decision Tree and Support Vector Machine (RBF kernel).

> **How to run in Google Colab:** upload the NSO `Data` folder (with the `stata` sub-folder) to Google Drive, mount Drive in the cell below and set `DATA_DIR` to that folder.

### How to run in Google Colab
1. Upload the NSO folder `Climate Change Survey-2022` to **My Drive** (keep it private; the NSO licence does not allow redistribution).
2. Run the first code cell and approve the Google Drive prompt.
3. If your folder is elsewhere, edit `DATA_DIR` in the setup cell so it points to `.../Data/stata`.
4. **Runtime → Run all** (about 7 minutes). Figures are saved to `figures/` and all numbers to `results.json`.


In [1]:
# ---- 1. Setup -------------------------------------------------------------
import os, json, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import StratifiedGroupKFold, GridSearchCV, cross_validate, train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, roc_curve, confusion_matrix, ConfusionMatrixDisplay, RocCurveDisplay,
                             classification_report)
from sklearn.inspection import permutation_importance
from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import cross_val_predict
# SMOTE comes from imbalanced-learn (pre-installed on Colab; installed here if missing)
try:
    import imblearn
except ImportError:
    import subprocess, sys; subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'imbalanced-learn'])
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
warnings.filterwarnings('ignore')

# Colab: mount Google Drive (skipped automatically when running locally)
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass
DATA_DIR = os.environ.get('NCCS_DATA', '/content/drive/MyDrive/Climate Change Survey-2022/Data/stata')
FIG_DIR = 'figures'; os.makedirs(FIG_DIR, exist_ok=True)
SEED = 42
np.random.seed(SEED)
sns.set_theme(style='whitegrid', context='notebook')
PAL = {'Low/moderate impact (0)': '#2E75B6', 'High impact (1)': '#C55A11'}
def save(name):
    plt.tight_layout(); plt.savefig(f'{FIG_DIR}/{name}.png', dpi=200, bbox_inches='tight'); plt.show()
print('Data folder:', DATA_DIR)

Data folder: /mnt/user-data/uploads/Climate Change Survey-2022/Data/stata


## 2. Load the NSO files
The survey is released as separate files linked by `psu` (sampling unit) and `hhld` (household number). We use the Stata versions because they keep the official value labels.

In [2]:
def load(name, labels=True):
    df = pd.read_stata(f'{DATA_DIR}/{name}.dta', convert_categoricals=labels)
    df.columns = [c.lower() for c in df.columns]
    return df

s01, s022, s04, s05 = load('S01'), load('S02_2'), load('S04'), load('S05')
s063 = load('S06_3', labels=False)          # disaster impact (long format: household x disaster type)
w    = load('Weight')
for n, d in [('S01', s01), ('S02_2', s022), ('S04', s04), ('S05', s05), ('S06_3', s063), ('Weight', w)]:
    print(f'{n:7} {d.shape[0]:>7,} rows x {d.shape[1]:>2} columns')

S01       6,508 rows x 21 columns
S02_2     6,508 rows x 44 columns
S04       6,508 rows x 25 columns
S05       6,508 rows x  6 columns
S06_3   123,652 rows x 12 columns
Weight    6,508 rows x 11 columns


## 3. Define the target variable (from the survey's own questions)
Section 6.3 of the questionnaire asks, for each of 19 climate-induced disaster types (flood, landslide, drought, …):
* **F17** – was your family affected in the past 25 years? (yes / no / not applicable)
* **F18** – if yes, how strong was the impact? (1 very low … 5 very high)

Almost every household (≈94 %) was affected by *some* disaster, so "affected vs not affected" is too imbalanced to be useful. We therefore classify **impact severity**:

**`high_impact = 1`** if the household rated the impact of at least one disaster as *High* or *Very high* (F18 ≥ 4), otherwise **0** (unaffected, or only very low–moderate impacts).

In [3]:
key = ['psu', 'hhld']
g = s063.groupby(key)
target = pd.DataFrame({
    'n_types_affected': g['f17'].apply(lambda s: (s == 1).sum()),
    'max_severity': g['f18'].max(),
}).reset_index()
target['any_affected'] = (target.n_types_affected > 0).astype(int)
target['high_impact'] = (target.max_severity >= 4).astype(int)
print('Affected by any disaster :', target.any_affected.value_counts(normalize=True).round(3).to_dict())
print('High-impact target       :', target.high_impact.value_counts().to_dict(),
      target.high_impact.value_counts(normalize=True).round(3).to_dict())

Affected by any disaster : {1: 0.937, 0: 0.063}
High-impact target       : {0: 3972, 1: 2536} {0: 0.61, 1: 0.39}


## 4. Build household-level features (leakage-safe)
Only variables describing the household, its location and its resources are used. All impact, loss, perception and adaptation-response sections (S06–S11) are **excluded** because they are recorded *after* or *because of* the disasters (target leakage). Identifiers and anonymised fields (names, phone, GPS – all blank in the release) are dropped. Caste/ethnicity is excluded on ethical grounds.

In [4]:
X = s01[key + ['prov', 'respsex', 'respage', 'a15', 'b01']].copy()
X = X.rename(columns={'prov': 'province', 'respsex': 'resp_sex', 'respage': 'resp_age',
                      'a15': 'years_in_community', 'b01': 'household_size'})
# education -> approximate years of schooling
edu_codes = load('S01', labels=False)[key + ['a14']]
edu_map = {**{i: i for i in range(0, 11)}, 11: 10, 12: 12, 13: 15, 14: 17, 15: 20, 16: 1, 17: 0}
edu_codes['resp_edu_years'] = edu_codes['a14'].map(edu_map)           # 90/91/98 -> NaN (imputed later)
X = X.merge(edu_codes[key + ['resp_edu_years']], on=key)

# housing, assets, livelihood, land (S02_2)
assets = ['tv', 'radio', 'landlinephone', 'ordinrymobile', 'smartmobile', 'computer', 'internet', 'carjeep',
          'electrcvehicl', 'motorcycle', 'electrcbike', 'bicycle', 'electrcfan', 'refrigerator', 'washingmachine', 'airconditnr']
h = s022[key + ['b10', 'b11', 'b12a', 'b13a', 'b14', 'b15', 'b16a', 'b18', 'c01'] + assets].copy()
h['asset_count'] = (s022[assets] == 'Have').sum(axis=1)
for a in assets: h[a] = (s022[a] == 'Have').astype(int)
h = h.rename(columns={'b10': 'house_ownership', 'b11': 'house_type', 'b12a': 'water_source', 'b13a': 'cooking_fuel',
                      'b14': 'lighting', 'b15': 'toilet', 'b16a': 'main_livelihood', 'b18': 'gets_remittance',
                      'c01': 'has_agri_holding'})
# owned land in 2079 BS converted from traditional units to hectares
ropani = s022.c12_1br * 508.72 + s022.c12_1ka * 31.80 + s022.c12_1dp * 7.95
bigha  = s022.c12_1br * 6772.63 + s022.c12_1ka * 338.63 + s022.c12_1dp * 16.93
area = np.where(s022.c12unit == 'Ropani', ropani, np.where(s022.c12unit == 'Bigha', bigha, 0.0))
h['land_ha'] = np.asarray(area, dtype=float) / 10000
X = X.merge(h, on=key)

# access to services, finance, social participation (S04)
s = s04[key + ['d01', 'd02', 'd03', 'd04', 'd07', 'd09', 'd10', 'd17', 'd11', 'd12', 'd13', 'd14', 'd15', 'd16']].rename(columns={
    'd01': 'coop_member', 'd02': 'coop_saving', 'd03': 'bank_saving', 'd04': 'took_loan', 'd07': 'community_org',
    'd09': 'tole_committee', 'd10': 'agri_service_received', 'd17': 'uses_farm_machinery',
    'd11': 'km_ward_office', 'd12': 'km_road', 'd13': 'km_health', 'd14': 'km_school', 'd15': 'km_market', 'd16': 'km_agri_centre'})
X = X.merge(s, on=key)

# income in the last 12 months (S05) – log-transformed because it is highly skewed
inc = s05[key].copy()
for c, n in [('e01', 'inc_crops'), ('e02', 'inc_livestock'), ('e03', 'inc_forest'), ('e04', 'inc_nonagri')]:
    inc['log_' + n] = np.log1p(s05[c])
inc['log_inc_total'] = np.log1p(s05[['e01', 'e02', 'e03', 'e04']].sum(axis=1))
X = X.merge(inc, on=key)

# location / design variables (Weight file)
w2 = w[['psu', 'hhld', 'ecobelt', 'urbrur', 'altgrp', 'riskrating']].copy()
w2[['psu', 'hhld']] = w2[['psu', 'hhld']].astype(int)
X = X.merge(w2.rename(columns={'ecobelt': 'eco_belt', 'urbrur': 'urban_rural', 'altgrp': 'altitude_group', 'riskrating': 'area_risk_rating'}), on=key)

data = X.merge(target[key + ['high_impact']], on=key)
for c in data.columns:
    if not pd.api.types.is_numeric_dtype(data[c]): data[c] = data[c].astype(object)
print('Final dataset:', data.shape, '| duplicated households:', data.duplicated(key).sum())

Final dataset: (6508, 59) | duplicated households: 0


In [5]:
y = data['high_impact'].values
groups = data['psu'].values
feat = [c for c in data.columns if c not in key + ['high_impact']]
cat_cols = [c for c in feat if not pd.api.types.is_numeric_dtype(data[c])]
num_cols = [c for c in feat if c not in cat_cols]
print(f'{len(feat)} features: {len(num_cols)} numerical, {len(cat_cols)} categorical')
summary = pd.DataFrame({'type': ['categorical' if c in cat_cols else 'numerical' for c in feat],
                        'missing_%': (data[feat].isna().mean() * 100).round(2).values,
                        'unique': data[feat].nunique().values}, index=feat)
summary.sort_values('missing_%', ascending=False).head(10)

56 features: 33 numerical, 23 categorical


,type,missing_%,unique
resp_edu_years,numerical,1.4,15
province,categorical,0.0,7
resp_age,numerical,0.0,51
resp_sex,categorical,0.0,2
years_in_community,numerical,0.0,71
household_size,numerical,0.0,21
house_ownership,categorical,0.0,4
house_type,categorical,0.0,4
water_source,categorical,0.0,9
cooking_fuel,categorical,0.0,6


## 5. Exploratory data analysis

In [6]:
lab = data['high_impact'].map({0: 'Low/moderate impact (0)', 1: 'High impact (1)'})
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
vc = lab.value_counts().reindex(PAL.keys())
ax[0].bar(vc.index, vc.values, color=list(PAL.values()))
for i, v in enumerate(vc.values): ax[0].text(i, v + 60, f'{v:,} ({v/len(lab):.0%})', ha='center')
ax[0].set_title('Target class distribution'); ax[0].set_ylabel('Households'); ax[0].set_ylim(0, vc.max() * 1.15)
order = ['Mountain', 'Hill', 'Tarai']
rate = data.groupby('eco_belt')['high_impact'].mean().reindex(order) * 100
ax[1].bar(order, rate.values, color='#548235')
for i, v in enumerate(rate.values): ax[1].text(i, v + 1, f'{v:.0f}%', ha='center')
ax[1].set_title('High-impact rate by ecological belt'); ax[1].set_ylabel('% of households'); ax[1].set_ylim(0, 100)
save('fig_target')

In [7]:
pr = data.groupby('province')['high_impact'].agg(['mean', 'size']).sort_values('mean')
fig, ax = plt.subplots(figsize=(7, 3.6))
ax.barh(pr.index, pr['mean'] * 100, color='#2E75B6')
for i, (m, n) in enumerate(zip(pr['mean'], pr['size'])): ax.text(m * 100 + 1, i, f'{m:.0%} (n={n})', va='center', fontsize=9)
ax.set_xlabel('% high-impact households'); ax.set_xlim(0, 100); ax.set_title('High-impact rate by province')
save('fig_province')

In [8]:
show = ['resp_age', 'household_size', 'resp_edu_years', 'asset_count', 'land_ha', 'log_inc_total', 'km_road', 'km_market']
fig, axes = plt.subplots(2, 4, figsize=(14, 6))
for ax, c in zip(axes.ravel(), show):
    v = data[c].clip(upper=data[c].quantile(0.99))
    sns.kdeplot(x=v, hue=lab, ax=ax, common_norm=False, fill=True, alpha=.35, palette=PAL, legend=(c == show[0]))
    ax.set_title(c); ax.set_xlabel('')
save('fig_distributions')

In [9]:
corr = data[num_cols + ['high_impact']].drop(columns=assets).corr()
plt.figure(figsize=(11, 9))
sns.heatmap(corr, cmap='RdBu_r', center=0, vmin=-1, vmax=1, square=True, cbar_kws={'shrink': .7}, xticklabels=True, yticklabels=True)
plt.title('Correlation matrix of numerical features and target')
save('fig_correlation')
top = corr['high_impact'].drop('high_impact').abs().sort_values(ascending=False).head(8)
print('Strongest linear correlations with the target:'); print(corr.loc[top.index, 'high_impact'].round(3))

Strongest linear correlations with the target:
asset_count         -0.232
log_inc_nonagri     -0.185
km_market            0.135
log_inc_crops        0.126
log_inc_total       -0.117
log_inc_livestock    0.115
km_ward_office       0.114
km_health            0.094
Name: high_impact, dtype: float64


In [10]:
rr = pd.crosstab(data['area_risk_rating'], data['high_impact'], normalize='index').reindex(['Very Low', 'Low', 'Moderate', 'High', 'Very High'])
print('High-impact share by NSO area risk rating:'); print((rr[1] * 100).round(1))

High-impact share by NSO area risk rating:
area_risk_rating
Very Low     39.5
Low          44.2
Moderate     30.2
High         20.0
Very High    29.2
Name: 1, dtype: float64


## 6. Train/test split grouped by sampling unit
The 20 households in each PSU share the same location and hazard exposure. A random split would put neighbours in both training and test sets, so the model could memorise locations and look better than it really is. We therefore hold out **whole PSUs** (≈20 %) using `StratifiedGroupKFold`, which also keeps the class balance similar.

In [11]:
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
train_idx, test_idx = next(sgkf.split(data[feat], y, groups))
Xtr, Xte = data.iloc[train_idx][feat], data.iloc[test_idx][feat]
ytr, yte = y[train_idx], y[test_idx]
gtr = groups[train_idx]
print(f'Train: {len(Xtr):,} households in {len(set(gtr))} PSUs | positive rate {ytr.mean():.3f}')
print(f'Test : {len(Xte):,} households in {len(set(groups[test_idx]))} PSUs | positive rate {yte.mean():.3f}')
print('PSUs shared between train and test:', len(set(gtr) & set(groups[test_idx])))

Train: 5,209 households in 261 PSUs | positive rate 0.390
Test : 1,299 households in 65 PSUs | positive rate 0.389
PSUs shared between train and test: 0


## 7. Pre-processing pipeline
All steps are learned on the training folds only (inside a scikit-learn `Pipeline`): median imputation + standard scaling for numerical features, most-frequent imputation + one-hot encoding for categorical features. Class imbalance (≈39 % positives) is handled with `class_weight='balanced'`.

In [12]:
pre = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())]), num_cols),
    ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')),
                      ('oh', OneHotEncoder(handle_unknown='ignore', min_frequency=20))]), cat_cols)])
pre.fit(Xtr); print('Features after encoding:', pre.transform(Xtr).shape[1])

Features after encoding: 115


## 8. Models and hyper-parameter tuning
Each model is tuned with a grid search using 5-fold `StratifiedGroupKFold` on the **training set only** (PSU-grouped), optimising the F1-score of the high-impact class.

In [13]:
models = {
    'Logistic Regression': (LogisticRegression(max_iter=5000, class_weight='balanced', solver='liblinear'),
                            {'clf__C': [0.01, 0.1, 1, 10], 'clf__penalty': ['l1', 'l2']}),
    'Decision Tree': (DecisionTreeClassifier(class_weight='balanced', random_state=SEED),
                      {'clf__max_depth': [3, 4, 5, 6, 8, 10], 'clf__min_samples_leaf': [10, 25, 50], 'clf__criterion': ['gini', 'entropy']}),
    'SVM (RBF)': (SVC(kernel='rbf', class_weight='balanced', probability=True, random_state=SEED),
                  {'clf__C': [0.1, 1, 10], 'clf__gamma': ['scale', 0.01, 0.001]}),
}
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
best, tuning = {}, {}
for name, (est, grid) in models.items():
    gs = GridSearchCV(Pipeline([('pre', pre), ('clf', est)]), grid, scoring='f1', cv=cv, n_jobs=-1)
    gs.fit(Xtr, ytr, groups=gtr)
    best[name] = gs.best_estimator_
    tuning[name] = {'best_params': {k.replace('clf__', ''): v for k, v in gs.best_params_.items()},
                    'cv_f1_mean': gs.best_score_, 'cv_f1_sd': gs.cv_results_['std_test_score'][gs.best_index_]}
    print(f'{name:20} best {tuning[name]["best_params"]} | CV F1 = {gs.best_score_:.3f} ± {tuning[name]["cv_f1_sd"]:.3f}')

Logistic Regression  best {'C': 0.01, 'penalty': 'l2'} | CV F1 = 0.641 ± 0.024


Decision Tree        best {'criterion': 'gini', 'max_depth': 6, 'min_samples_leaf': 25} | CV F1 = 0.609 ± 0.031


SVM (RBF)            best {'C': 1, 'gamma': 'scale'} | CV F1 = 0.646 ± 0.031


## 9. Cross-validation stability on the training set

In [14]:
cvres = {}
for name, m in best.items():
    r = cross_validate(m, Xtr, ytr, groups=gtr, cv=cv, n_jobs=-1, scoring=['accuracy', 'precision', 'recall', 'f1', 'roc_auc'])
    cvres[name] = {k.replace('test_', ''): (v.mean(), v.std()) for k, v in r.items() if k.startswith('test_')}
cv_table = pd.DataFrame({n: {k: f'{m:.3f} ± {s:.3f}' for k, (m, s) in d.items()} for n, d in cvres.items()}).T
cv_table

,accuracy,precision,recall,f1,roc_auc
Logistic Regression,0.689 ± 0.024,0.584 ± 0.030,0.713 ± 0.034,0.641 ± 0.024,0.744 ± 0.022
Decision Tree,0.646 ± 0.025,0.537 ± 0.029,0.712 ± 0.077,0.609 ± 0.031,0.707 ± 0.033
SVM (RBF),0.708 ± 0.024,0.615 ± 0.033,0.682 ± 0.051,0.646 ± 0.031,0.767 ± 0.026


## 10. Final evaluation on the held-out test PSUs

In [15]:
rows, probs = [], {}
for name, m in best.items():
    p = m.predict(Xte); pr_ = m.predict_proba(Xte)[:, 1]; probs[name] = pr_
    rows.append({'Model': name, 'Accuracy': accuracy_score(yte, p), 'Precision': precision_score(yte, p),
                 'Recall': recall_score(yte, p), 'F1': f1_score(yte, p), 'ROC-AUC': roc_auc_score(yte, pr_),
                 'Macro F1': f1_score(yte, p, average='macro')})
results = pd.DataFrame(rows).set_index('Model').round(3)
# simple reference points: any useful model must beat these
base_rows = []
for bname, strat in [('Baseline: always "low/moderate"', 'most_frequent'), ('Baseline: always "high"', 'constant'),
                     ('Baseline: random (stratified)', 'stratified')]:
    dm = DummyClassifier(strategy=strat, constant=1 if strat == 'constant' else None, random_state=SEED).fit(Xtr, ytr)
    p = dm.predict(Xte); pr_ = dm.predict_proba(Xte)[:, 1] if strat != 'constant' else np.ones(len(yte))
    base_rows.append({'Model': bname, 'Accuracy': accuracy_score(yte, p), 'Precision': precision_score(yte, p, zero_division=0),
                      'Recall': recall_score(yte, p), 'F1': f1_score(yte, p), 'ROC-AUC': roc_auc_score(yte, pr_) if len(set(pr_)) > 1 else 0.5,
                      'Macro F1': f1_score(yte, p, average='macro')})
baselines = pd.DataFrame(base_rows).set_index('Model').round(3)
pd.concat([results, baselines])

,Accuracy,Precision,Recall,F1,ROC-AUC,Macro F1
Model,,,,,,
Logistic Regression,0.654,0.545,0.667,0.600,0.704,0.648
Decision Tree,0.637,0.524,0.747,0.616,0.719,0.636
SVM (RBF),0.663,0.559,0.628,0.591,0.724,0.652
"Baseline: always ""low/moderate""",0.611,0.000,0.000,0.000,0.500,0.379
"Baseline: always ""high""",0.389,0.389,1.000,0.560,0.500,0.280
Baseline: random (stratified),0.545,0.416,0.422,0.419,0.523,0.523


In [16]:
for name, m in best.items():
    print('=' * 12, name); print(classification_report(yte, m.predict(Xte), target_names=['Low/moderate', 'High'], digits=3))

============ Logistic Regression
              precision    recall  f1-score   support

Low/moderate      0.753     0.646     0.696       794
        High      0.545     0.667     0.600       505

    accuracy                          0.654      1299
   macro avg      0.649     0.657     0.648      1299
weighted avg      0.672     0.654     0.658      1299

============ Decision Tree
              precision    recall  f1-score   support

Low/moderate      0.779     0.568     0.657       794
        High      0.524     0.747     0.616       505

    accuracy                          0.637      1299
   macro avg      0.651     0.657     0.636      1299
weighted avg      0.680     0.637     0.641      1299

============ SVM (RBF)


              precision    recall  f1-score   support

Low/moderate      0.743     0.685     0.713       794
        High      0.559     0.628     0.591       505

    accuracy                          0.663      1299
   macro avg      0.651     0.656     0.652      1299
weighted avg      0.672     0.663     0.666      1299



In [17]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
for ax, (name, m) in zip(axes, best.items()):
    ConfusionMatrixDisplay(confusion_matrix(yte, m.predict(Xte)), display_labels=['Low/mod.', 'High']).plot(ax=ax, cmap='Blues', colorbar=False)
    ax.set_title(name); ax.grid(False)
save('fig_confusion')

In [18]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
results[['Accuracy', 'Precision', 'Recall', 'F1', 'ROC-AUC']].T.plot.bar(ax=ax[0], color=['#2E75B6', '#548235', '#C55A11'], rot=0, width=.8)
ax[0].set_ylim(0, 1); ax[0].set_title('Test-set performance'); ax[0].legend(loc='lower right', fontsize=8)
for name, c in zip(best, ['#2E75B6', '#548235', '#C55A11']):
    fpr, tpr, _ = roc_curve(yte, probs[name])
    ax[1].plot(fpr, tpr, color=c, label=f'{name} (AUC = {roc_auc_score(yte, probs[name]):.3f})')
ax[1].legend(loc='lower right', fontsize=8); ax[1].set_xlabel('False positive rate'); ax[1].set_ylabel('True positive rate')
ax[1].plot([0, 1], [0, 1], 'k--', lw=1); ax[1].set_title('ROC curves (test set)')
save('fig_performance')

## 11. Which features matter? (permutation importance on the test set)

In [19]:
imp = {}
for name, m in best.items():
    r = permutation_importance(m, Xte, yte, scoring='f1', n_repeats=10, random_state=SEED, n_jobs=-1)
    imp[name] = pd.Series(r.importances_mean, index=feat)
imp = pd.DataFrame(imp)
top = imp.mean(axis=1).sort_values(ascending=False).head(12).index
imp.loc[top[::-1]].plot.barh(figsize=(8, 5.5), color=['#2E75B6', '#548235', '#C55A11'], width=.8)
plt.xlabel('Mean drop in test F1 when feature is shuffled'); plt.title('Top-12 features (permutation importance)')
save('fig_importance')
imp.loc[top].round(4)

,Logistic Regression,Decision Tree,SVM (RBF)
province,0.0308,0.0750,0.0393
agri_service_received,0.0090,0.0347,0.0059
house_type,0.0020,0.0333,0.0045
land_ha,-0.0008,0.0184,0.0085
has_agri_holding,0.0111,0.0000,0.0113
area_risk_rating,0.0105,0.0000,0.0112
log_inc_crops,0.0063,0.0077,0.0064
main_livelihood,0.0067,0.0053,0.0070
km_market,0.0039,0.0089,0.0033
community_org,0.0098,0.0000,0.0059


In [20]:
dt = best['Decision Tree']
names = dt.named_steps['pre'].get_feature_names_out()
plt.figure(figsize=(16, 6))
plot_tree(dt.named_steps['clf'], max_depth=2, feature_names=[n.split('__')[1] for n in names],
          class_names=['Low/mod.', 'High'], filled=True, impurity=False, proportion=True, fontsize=9)
save('fig_tree')
lr = best['Logistic Regression']
coef = pd.Series(lr.named_steps['clf'].coef_[0], index=[n.split('__')[1] for n in lr.named_steps['pre'].get_feature_names_out()])
print('Largest Logistic Regression coefficients (standardised):')
print(coef.reindex(coef.abs().sort_values(ascending=False).index).head(10).round(3))

Largest Logistic Regression coefficients (standardised):
province_Lumbini                        -0.561
province_Koshi                           0.295
house_type_Permanent                    -0.256
house_type_Temporary                     0.252
toilet_Ordinary                          0.247
province_Madhesh                        -0.237
main_livelihood_Agriculture              0.230
eco_belt_Hill                           -0.201
province_Gandaki                         0.180
toilet_Flush toilet (Public sewarage)   -0.173
dtype: float64


## 12. Fairness check – performance by ecological belt

In [21]:
belt = data.iloc[test_idx]['eco_belt'].values
frows = []
for name, m in best.items():
    p = m.predict(Xte)
    for b in ['Mountain', 'Hill', 'Tarai']:
        k = belt == b
        frows.append({'Model': name, 'Belt': b, 'n': int(k.sum()), 'High-impact rate': yte[k].mean(),
                      'Recall': recall_score(yte[k], p[k], zero_division=0), 'F1': f1_score(yte[k], p[k], zero_division=0),
                      'Accuracy': accuracy_score(yte[k], p[k])})
fair = pd.DataFrame(frows).round(3)
fair.pivot(index='Belt', columns='Model', values='F1').reindex(['Mountain', 'Hill', 'Tarai']).plot.bar(
    figsize=(7, 3.6), rot=0, color=['#548235', '#2E75B6', '#C55A11'], width=.8)
plt.ylim(0, 1); plt.ylabel('Test F1 (high-impact class)'); plt.title('Performance by ecological belt')
save('fig_fairness')
fair

,Model,Belt,n,High-impact rate,Recall,F1,Accuracy
0,Logistic Regression,Mountain,180,0.456,0.915,0.641,0.533
1,Logistic Regression,Hill,520,0.363,0.635,0.606,0.700
2,Logistic Regression,Tarai,599,0.391,0.607,0.576,0.651
3,Decision Tree,Mountain,180,0.456,0.866,0.609,0.494
4,Decision Tree,Hill,520,0.363,0.825,0.664,0.696
5,Decision Tree,Tarai,599,0.391,0.641,0.575,0.629
6,SVM (RBF),Mountain,180,0.456,0.780,0.615,0.556
7,SVM (RBF),Hill,520,0.363,0.582,0.603,0.721
8,SVM (RBF),Tarai,599,0.391,0.611,0.573,0.644


## 13. Alternative ways of handling class imbalance: SMOTE and threshold tuning
The main models use **class weights**. Two alternatives are tested with the same tuned hyper-parameters and the same grouped split:
* **SMOTE** (Synthetic Minority Over-sampling Technique) creates synthetic high-impact households by interpolating between real ones. It is applied **inside the pipeline after pre-processing and only on training folds**, so the test set is never resampled.
* **Threshold tuning** keeps the class-weighted models but changes the decision cut-off (default 0.5). The threshold that maximises F1 is chosen from **out-of-fold predictions on the training set** (grouped 5-fold CV) and then applied unchanged to the test set.

In [22]:
imb_rows, thresholds, oof_store = [], {}, {}
for name, m in best.items():
    params = {k: v for k, v in m.named_steps['clf'].get_params().items()}
    # --- (a) SMOTE: same classifier settings but without class weights
    clf_s = clone(m.named_steps['clf']).set_params(class_weight=None)
    smote_pipe = ImbPipeline([('pre', clone(pre)), ('smote', SMOTE(random_state=SEED)), ('clf', clf_s)])
    cv_f1 = cross_validate(smote_pipe, Xtr, ytr, groups=gtr, cv=cv, scoring='f1', n_jobs=-1)['test_score']
    smote_pipe.fit(Xtr, ytr)
    p = smote_pipe.predict(Xte); pr_ = smote_pipe.predict_proba(Xte)[:, 1]
    imb_rows.append({'Model': name, 'Strategy': 'SMOTE', 'Threshold': 0.5, 'CV F1': cv_f1.mean(),
                     'Accuracy': accuracy_score(yte, p), 'Precision': precision_score(yte, p), 'Recall': recall_score(yte, p),
                     'F1': f1_score(yte, p), 'ROC-AUC': roc_auc_score(yte, pr_)})
    # --- (b) threshold tuning on out-of-fold training probabilities
    oof = cross_val_predict(clone(m), Xtr, ytr, groups=gtr, cv=cv, method='predict_proba', n_jobs=-1)[:, 1]
    grid_t = np.round(np.arange(0.20, 0.81, 0.01), 2)
    f1s = [f1_score(ytr, (oof >= t).astype(int)) for t in grid_t]
    t_best = float(grid_t[int(np.argmax(f1s))]); thresholds[name] = t_best; oof_store[name] = (grid_t, f1s)
    p = (probs[name] >= t_best).astype(int)
    imb_rows.append({'Model': name, 'Strategy': 'Class weights + tuned threshold', 'Threshold': t_best, 'CV F1': max(f1s),
                     'Accuracy': accuracy_score(yte, p), 'Precision': precision_score(yte, p), 'Recall': recall_score(yte, p),
                     'F1': f1_score(yte, p), 'ROC-AUC': roc_auc_score(yte, probs[name])})
    # --- reference: class weights at the default 0.5 threshold (main results)
    imb_rows.append({'Model': name, 'Strategy': 'Class weights (main)', 'Threshold': 0.5, 'CV F1': tuning[name]['cv_f1_mean'],
                     **{k: results.loc[name, k] for k in ['Accuracy', 'Precision', 'Recall', 'F1', 'ROC-AUC']}})
imb = pd.DataFrame(imb_rows)
imb['Strategy'] = pd.Categorical(imb['Strategy'], ['Class weights (main)', 'SMOTE', 'Class weights + tuned threshold'])
imb = imb.sort_values(['Model', 'Strategy']).set_index(['Model', 'Strategy']).round(3)
print('Thresholds chosen on training out-of-fold predictions:', thresholds)
imb

Thresholds chosen on training out-of-fold predictions: {'Logistic Regression': 0.47, 'Decision Tree': 0.37, 'SVM (RBF)': 0.29}


Threshold  CV F1  \
Model               Strategy                                            
Decision Tree       Class weights (main)                  0.50  0.609   
                    SMOTE                                 0.50  0.584   
                    Class weights + tuned threshold       0.37  0.625   
Logistic Regression Class weights (main)                  0.50  0.641   
                    SMOTE                                 0.50  0.637   
                    Class weights + tuned threshold       0.47  0.644   
SVM (RBF)           Class weights (main)                  0.50  0.646   
                    SMOTE                                 0.50  0.628   
                    Class weights + tuned threshold       0.29  0.653   

                                                     Accuracy  Precision  \
Model               Strategy                                               
Decision Tree       Class weights (main)                0.637      0.524   
                    SMOTE                               0.644      0.542   
                    Class weights + tuned threshold     0.630      0.515   
Logistic Regression Class weights (main)                0.654      0.545   
                    SMOTE                               0.650      0.540   
                    Class weights + tuned threshold     0.644      0.531   
SVM (RBF)           Class weights (main)                0.663      0.559   
                    SMOTE                               0.649      0.548   
                    Class weights + tuned threshold     0.656      0.540   

                                                     Recall     F1  ROC-AUC  
Model               Strategy                                                 
Decision Tree       Class weights (main)              0.747  0.616    0.719  
                    SMOTE                             0.533  0.537    0.688  
                    Class weights + tuned threshold   0.846  0.640    0.719  
Logistic Regression Class weights (main)              0.667  0.600    0.704  
                    SMOTE                             0.669  0.598    0.697  
                    Class weights + tuned threshold   0.707  0.607    0.704  
SVM (RBF)           Class weights (main)              0.628  0.591    0.724  
                    SMOTE                             0.558  0.553    0.715  
                    Class weights + tuned threshold   0.770  0.635    0.724

In [23]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for name, c in zip(best, ['#2E75B6', '#548235', '#C55A11']):
    gt, f = oof_store[name]
    ax[0].plot(gt, f, color=c, label=f'{name} (best t = {thresholds[name]:.2f})')
    ax[0].axvline(thresholds[name], color=c, ls=':', lw=1)
ax[0].set_xlabel('Decision threshold'); ax[0].set_ylabel('Out-of-fold F1 (training set)')
ax[0].set_title('Choosing the decision threshold'); ax[0].legend(fontsize=8)
piv = imb['F1'].unstack('Strategy')[['Class weights (main)', 'SMOTE', 'Class weights + tuned threshold']]
piv.plot.bar(ax=ax[1], rot=0, color=['#7F7F7F', '#8FAADC', '#C55A11'], width=.8)
ax[1].axhline(baselines.loc['Baseline: always "high"', 'F1'], color='k', ls='--', lw=1, label='Baseline: always "high"')
ax[1].set_ylim(0.4, 0.75); ax[1].set_ylabel('Test F1 (high-impact class)'); ax[1].set_title('Imbalance strategies on the test set')
ax[1].legend(fontsize=8, loc='upper left')
save('fig_imbalance')

## 14. Why the grouped split matters (leakage check)
For comparison only, the best model of each type is re-trained on a **random** 80/20 split that ignores PSUs.

In [24]:
Xa, Xb, ya, yb = train_test_split(data[feat], y, test_size=0.2, stratify=y, random_state=SEED)
leak = {}
for name, m in best.items():
    mm = clone(m); mm.fit(Xa, ya)          # clone = fresh copy, the tuned test model is not touched
    leak[name] = {'Random split F1': f1_score(yb, mm.predict(Xb)), 'Grouped split F1': results.loc[name, 'F1']}
leak = pd.DataFrame(leak).T.round(3); leak['Difference'] = (leak['Random split F1'] - leak['Grouped split F1']).round(3)
leak

,Random split F1,Grouped split F1,Difference
Logistic Regression,0.650,0.600,0.050
Decision Tree,0.644,0.616,0.028
SVM (RBF),0.675,0.591,0.084


In [25]:
out = {'n_households': int(len(data)), 'n_features': len(feat), 'n_num': len(num_cols), 'n_cat': len(cat_cols),
       'n_encoded': int(pre.transform(Xtr).shape[1]), 'pos_rate': float(y.mean()), 'n_pos': int(y.sum()),
       'any_affected_rate': float(target.any_affected.mean()),
       'train': [int(len(Xtr)), int(len(set(gtr))), float(ytr.mean())],
       'test': [int(len(Xte)), int(len(set(groups[test_idx]))), float(yte.mean())],
       'tuning': tuning, 'cv': {n: {k: list(v) for k, v in d.items()} for n, d in cvres.items()},
       'test_results': results.to_dict(orient='index'),
       'confusion': {n: confusion_matrix(yte, m.predict(Xte)).tolist() for n, m in best.items()},
       'importance_top': imp.loc[top].round(4).to_dict(orient='index'),
       'lr_coef_top': coef.reindex(coef.abs().sort_values(ascending=False).index).head(10).round(3).to_dict(),
       'fairness': fair.to_dict(orient='records'), 'leakage': leak.to_dict(orient='index'),
       'rate_by_belt': data.groupby('eco_belt')['high_impact'].mean().round(3).to_dict(),
       'rate_by_prov': data.groupby('province')['high_impact'].mean().round(3).to_dict(),
       'rate_by_risk': (rr[1]).round(3).to_dict(),
       'corr_top': corr.loc[corr['high_impact'].drop('high_impact').abs().sort_values(ascending=False).head(8).index, 'high_impact'].round(3).to_dict(),
       'baselines': baselines.to_dict(orient='index'),
       'imbalance': {f'{a} | {b}': v for (a, b), v in imb.to_dict(orient='index').items()},
       'thresholds': thresholds,
       'missing_top': summary.sort_values('missing_%', ascending=False).head(6)['missing_%'].to_dict()}
json.dump(out, open('results.json', 'w'), indent=1, default=float)
print('Saved results.json and figures/')

Saved results.json and figures/
